# Locy Case Study: Drug Repurposing via Biomedical Knowledge Graph

This case study demonstrates probabilistic graph reasoning for **drug repurposing** —
identifying new therapeutic uses for existing approved drugs.

**Key Locy features demonstrated:**
- `ALONG` — recursive pathway traversal through protein-protein interaction networks
- `FOLD MNOR` — noisy-OR aggregation of independent mechanistic pathway evidence
- `similar_to` — molecular fingerprint similarity for structural analogue discovery
- `IS NOT` — exclusion of known approved indications
- `ASSUME` — counterfactual "what-if" binding simulation
- `ABDUCE` — minimal evidence search for target disease candidates
- `EXPLAIN RULE` — derivation tree inspection for explainable drug-disease predictions

## How To Read This Notebook

- Section 1 sets up helpers, locates data files, and creates an isolated temporary database.
- Section 2 loads CSV snapshots and builds a focus cohort around 4 drugs with known repurposing stories.
- Section 3 defines the graph schema (labels, properties, edge types).
- Section 4 ingests all nodes and edges.
- Section 5 declares the full Locy program (ALONG, FOLD MNOR, IS NOT, similar_to) and evaluates it.
- Sections 6-8 demonstrate EXPLAIN RULE, ASSUME, and ABDUCE.
- Section 9 describes expected outputs; Section 10 runs build-time assertions; Section 11 cleans up.

## 1) Setup & Data Discovery

Loads helpers, locates prepared CSV data files, and creates an isolated temporary database.

In [ ]:
from pathlib import Path
from pprint import pprint
import csv
import json
import os
import shutil
import tempfile

import uni_db


def _read_csv(path: Path) -> list[dict[str, str]]:
    with path.open('r', encoding='utf-8', newline='') as f:
        return list(csv.DictReader(f))


def _esc(value: str) -> str:
    return str(value).replace('\\', '\\\\').replace("'", "\\'")


def _f(value: str) -> float:
    return float(value) if value not in ('', None) else 0.0


def _vec(value: str) -> list[float]:
    return [float(x) for x in json.loads(value)]


def _norm_key(key: object) -> str:
    s = str(key)
    if s.startswith('Variable("') and s.endswith('")'):
        return s[len('Variable("'):-2]
    return s


def _norm_rows(rows: list[dict[object, object]]) -> list[dict[str, object]]:
    return [{_norm_key(k): v for k, v in row.items()} for row in rows]


def _print_tree(node, depth=0, max_depth=3, max_children=5):
    indent = '  ' * depth
    print(f"{indent}- rule={node.get('rule')}, clause={node.get('clause_index')}, bindings={node.get('bindings', {})}")
    if depth >= max_depth:
        return
    children = node.get('children', [])
    for child in children[:max_children]:
        _print_tree(child, depth + 1, max_depth=max_depth, max_children=max_children)
    if len(children) > max_children:
        print(f"{indent}  ... {len(children) - max_children} more child derivations")


_default_candidates = [
    Path('docs/examples/data/locy_drug_repurposing'),
    Path('website/docs/examples/data/locy_drug_repurposing'),
    Path('examples/data/locy_drug_repurposing'),
    Path('../data/locy_drug_repurposing'),
]
if 'LOCY_DATA_DIR' in os.environ:
    DATA_DIR = Path(os.environ['LOCY_DATA_DIR']).resolve()
else:
    DATA_DIR = next(
        (p.resolve() for p in _default_candidates if (p / 'drugs.csv').exists()),
        _default_candidates[0].resolve(),
    )
if not (DATA_DIR / 'drugs.csv').exists():
    raise FileNotFoundError(
        'Expected data under docs/examples/data/locy_drug_repurposing. '
        'Run from website/ (or repo root) or set LOCY_DATA_DIR.'
    )
DB_DIR = tempfile.mkdtemp(prefix='uni_locy_drug_')
db = uni_db.Uni.open(DB_DIR)
session = db.session()
print('DATA_DIR:', DATA_DIR)
print('DB_DIR:', DB_DIR)

## 2) Load Data & Build Focus Cohort

Loads deterministic CSV snapshots and selects a focus cohort around 4 drugs with known repurposing stories.
PPI neighbors are expanded 3 hops to capture multi-step mechanistic pathways.

In [ ]:
drugs = _read_csv(DATA_DIR / 'drugs.csv')
proteins = _read_csv(DATA_DIR / 'proteins.csv')
diseases = _read_csv(DATA_DIR / 'diseases.csv')
side_effects = _read_csv(DATA_DIR / 'side_effects.csv')
binds = _read_csv(DATA_DIR / 'binds.csv')
interacts = _read_csv(DATA_DIR / 'interacts.csv')
associated_with = _read_csv(DATA_DIR / 'associated_with.csv')
indicated_for = _read_csv(DATA_DIR / 'indicated_for.csv')
causes_adr = _read_csv(DATA_DIR / 'causes_adr.csv')
notebook_cases = _read_csv(DATA_DIR / 'notebook_cases.csv')

# Focus on 4 drugs with known repurposing stories
focus_drug_ids = {r['drug_id'] for r in notebook_cases}
focus_drugs = [r for r in drugs if r['drug_id'] in focus_drug_ids]

# Include all proteins, diseases, side effects (small dataset)
# Filter edges to focus drugs
focus_binds = [r for r in binds if r['drug_id'] in focus_drug_ids]
focus_protein_ids = {r['protein_id'] for r in focus_binds}
# Also include proteins reachable via PPI from focus targets
for _ in range(2):  # 3-hop expansion
    new_pids = set()
    for r in interacts:
        if r['src_protein_id'] in focus_protein_ids:
            new_pids.add(r['dst_protein_id'])
        if r['dst_protein_id'] in focus_protein_ids:
            new_pids.add(r['src_protein_id'])
    focus_protein_ids.update(new_pids)

focus_proteins = [r for r in proteins if r['protein_id'] in focus_protein_ids]
focus_interacts = [r for r in interacts if r['src_protein_id'] in focus_protein_ids and r['dst_protein_id'] in focus_protein_ids]
focus_assoc = [r for r in associated_with if r['protein_id'] in focus_protein_ids]
focus_disease_ids = {r['disease_id'] for r in focus_assoc}
focus_diseases = [r for r in diseases if r['disease_id'] in focus_disease_ids]
focus_indicated = [r for r in indicated_for if r['drug_id'] in focus_drug_ids]
focus_adr = [r for r in causes_adr if r['drug_id'] in focus_drug_ids]

print('focus drugs:', len(focus_drugs))
print('focus proteins:', len(focus_proteins))
print('focus diseases:', len(focus_diseases))
print('focus binds:', len(focus_binds))
print('focus PPI interactions:', len(focus_interacts))
print('focus gene-disease assoc:', len(focus_assoc))
print('focus indications:', len(focus_indicated))

## 3) Define Schema

Defines explicit labels, typed properties, vector dimensions, and edge types before ingest.

In [ ]:
(
    db.schema()
    .label('Drug')
        .property('drug_id', 'string')
        .property('name', 'string')
        .property('drug_class', 'string')
        .property('approval_status', 'string')
        .vector('fingerprint', 4)
    .done()
    .label('Protein')
        .property('protein_id', 'string')
        .property('name', 'string')
        .property('gene_symbol', 'string')
        .property('family', 'string')
    .done()
    .label('Disease')
        .property('disease_id', 'string')
        .property('name', 'string')
        .property('therapeutic_area', 'string')
    .done()
    .label('SideEffect')
        .property('se_id', 'string')
        .property('name', 'string')
        .property('severity', 'string')
    .done()
    .edge_type('BINDS', ['Drug'], ['Protein'])
        .property('affinity_nm', 'float64')
        .property('confidence', 'float64')
    .done()
    .edge_type('INTERACTS', ['Protein'], ['Protein'])
        .property('string_score', 'float64')
    .done()
    .edge_type('ASSOCIATED_WITH', ['Protein'], ['Disease'])
        .property('gda_score', 'float64')
    .done()
    .edge_type('INDICATED_FOR', ['Drug'], ['Disease'])
        .property('evidence', 'string')
    .done()
    .edge_type('CAUSES_ADR', ['Drug'], ['SideEffect'])
        .property('frequency', 'float64')
    .done()
    .apply()
)
print('Schema created')

## 4) Ingest Graph Facts

Creates all nodes and edges from the focus cohort CSVs using Cypher CREATE/MATCH statements.

In [ ]:
tx = session.tx()

for row in focus_drugs:
    fp = _vec(row['fingerprint'])
    tx.execute(
        f"CREATE (:Drug {{drug_id: '{_esc(row['drug_id'])}', name: '{_esc(row['name'])}', "
        f"drug_class: '{_esc(row['drug_class'])}', approval_status: '{_esc(row['approval_status'])}', "
        f"fingerprint: {fp}}})"
    )

for row in focus_proteins:
    tx.execute(
        f"CREATE (:Protein {{protein_id: '{_esc(row['protein_id'])}', name: '{_esc(row['name'])}', "
        f"gene_symbol: '{_esc(row['gene_symbol'])}', family: '{_esc(row['family'])}'}})"
    )

for row in focus_diseases:
    tx.execute(
        f"CREATE (:Disease {{disease_id: '{_esc(row['disease_id'])}', name: '{_esc(row['name'])}', "
        f"therapeutic_area: '{_esc(row['therapeutic_area'])}'}})"
    )

for row in side_effects:
    tx.execute(
        f"CREATE (:SideEffect {{se_id: '{_esc(row['se_id'])}', name: '{_esc(row['name'])}', "
        f"severity: '{_esc(row['severity'])}'}})"
    )

for row in focus_binds:
    tx.execute(
        f"MATCH (d:Drug {{drug_id: '{_esc(row['drug_id'])}'}}), (p:Protein {{protein_id: '{_esc(row['protein_id'])}'}}) "
        f"CREATE (d)-[:BINDS {{affinity_nm: {_f(row['affinity_nm'])}, confidence: {_f(row['confidence'])}}}]->(p)"
    )

for row in focus_interacts:
    tx.execute(
        f"MATCH (p1:Protein {{protein_id: '{_esc(row['src_protein_id'])}'}}), (p2:Protein {{protein_id: '{_esc(row['dst_protein_id'])}'}}) "
        f"CREATE (p1)-[:INTERACTS {{string_score: {_f(row['string_score'])}}}]->(p2)"
    )

for row in focus_assoc:
    tx.execute(
        f"MATCH (p:Protein {{protein_id: '{_esc(row['protein_id'])}'}}), (dis:Disease {{disease_id: '{_esc(row['disease_id'])}'}}) "
        f"CREATE (p)-[:ASSOCIATED_WITH {{gda_score: {_f(row['gda_score'])}}}]->(dis)"
    )

for row in focus_indicated:
    tx.execute(
        f"MATCH (d:Drug {{drug_id: '{_esc(row['drug_id'])}'}}), (dis:Disease {{disease_id: '{_esc(row['disease_id'])}'}}) "
        f"CREATE (d)-[:INDICATED_FOR {{evidence: '{_esc(row['evidence'])}'}}]->(dis)"
    )

for row in focus_adr:
    tx.execute(
        f"MATCH (d:Drug {{drug_id: '{_esc(row['drug_id'])}'}}), (se:SideEffect {{se_id: '{_esc(row['se_id'])}'}}) "
        f"CREATE (d)-[:CAUSES_ADR {{frequency: {_f(row['frequency'])}}}]->(se)"
    )

tx.commit()

counts = session.query("""
MATCH (d:Drug) WITH count(*) AS drugs
MATCH (p:Protein) WITH drugs, count(*) AS proteins
MATCH (dis:Disease) WITH drugs, proteins, count(*) AS diseases
MATCH (se:SideEffect) WITH drugs, proteins, diseases, count(*) AS side_effects
MATCH ()-[b:BINDS]->() WITH drugs, proteins, diseases, side_effects, count(*) AS binds
MATCH ()-[i:INTERACTS]->() WITH drugs, proteins, diseases, side_effects, binds, count(*) AS interactions
MATCH ()-[a:ASSOCIATED_WITH]->()
RETURN drugs, proteins, diseases, side_effects, binds, interactions, count(a) AS associations
""")
print('Graph counts:')
pprint(counts[0])

## 5) Baseline Locy Program

The core probabilistic reasoning pipeline:

1. **ppi_reach** (base + recursive) — `ALONG` traverses Protein -> PPI -> ... -> Disease, accumulating strength and hop count. `BEST BY` keeps the strongest shortest path.
2. **repurposing_signal** — Joins Drug->Protein binding with ppi_reach, then `FOLD MNOR` aggregates independent pathway strengths per drug-disease pair using noisy-OR (probability of at least one pathway being real).
3. **known_indication** — captures existing approved drug-disease pairs.
4. **novel_candidate** — `IS NOT` filters out known indications, leaving only repurposing candidates.
5. **structural_analogue** — `similar_to` on molecular fingerprint vectors finds drugs with similar structure that share target diseases.

In [ ]:
program = r'''
CREATE RULE ppi_reach AS
  MATCH (p:Protein)-[a:ASSOCIATED_WITH]->(dis:Disease)
  YIELD KEY p, KEY dis, a.gda_score AS strength

CREATE RULE ppi_reach AS
  MATCH (p:Protein)-[i:INTERACTS]->(p2:Protein)-[a:ASSOCIATED_WITH]->(dis:Disease)
  YIELD KEY p, KEY dis, i.string_score * a.gda_score AS strength

CREATE RULE ppi_reach AS
  MATCH (p:Protein)-[i1:INTERACTS]->(p2:Protein)-[i2:INTERACTS]->(p3:Protein)-[a:ASSOCIATED_WITH]->(dis:Disease)
  YIELD KEY p, KEY dis, i1.string_score * i2.string_score * a.gda_score AS strength

CREATE RULE drug_pathway AS
  MATCH (d:Drug)-[b:BINDS]->(p:Protein)
  WHERE p IS ppi_reach TO dis
  YIELD KEY d, KEY dis, b.confidence * strength AS pathway_score

CREATE RULE repurposing_signal AS
  MATCH (d:Drug)
  WHERE d IS drug_pathway TO dis
  FOLD evidence = MNOR(pathway_score)
  YIELD KEY d, KEY dis, evidence

CREATE RULE known_indication AS
  MATCH (d:Drug)-[:INDICATED_FOR]->(dis:Disease)
  YIELD KEY d, KEY dis

CREATE RULE novel_candidate AS
  MATCH (d:Drug)
  WHERE d IS repurposing_signal TO dis, d IS NOT known_indication TO dis
  YIELD KEY d, KEY dis, evidence

CREATE RULE structural_analogue AS
  MATCH (d1:Drug), (d2:Drug)-[:INDICATED_FOR]->(dis:Disease)
  WHERE d1 <> d2
  YIELD KEY d1, KEY dis, similar_to(d1.fingerprint, d2.fingerprint) AS analogy

QUERY novel_candidate WHERE d = d RETURN d.name AS drug, dis.name AS disease, evidence ORDER BY evidence DESC
QUERY structural_analogue WHERE analogy >= 0.5 RETURN d1.name AS drug, dis.name AS disease, analogy ORDER BY analogy DESC LIMIT 10
'''

baseline_out = session.locy_with(program).with_config({'max_iterations': 400, 'timeout_secs': 180.0}).run()
stats = baseline_out.stats
print('Iterations:', stats.total_iterations)
print('Strata:', stats.strata_evaluated)

novel_rows = []
analogue_rows = []
for i, cmd in enumerate(baseline_out.command_results, start=1):
    print(f'\nCommand #{i}:', cmd.command_type)
    rows = _norm_rows(cmd.rows)
    print('rows:', len(rows))
    pprint(rows[:8])
    if rows and 'evidence' in rows[0]:
        novel_rows = rows
    if rows and 'analogy' in rows[0]:
        analogue_rows = rows

for row in novel_rows:
    e = float(row['evidence'])
    assert 0.0 <= e <= 1.0, f'MNOR score out of range: {e}'
print(f'\nAll {len(novel_rows)} novel candidate scores in [0, 1]')


## 6) EXPLAIN RULE

Inspects the derivation tree behind a specific drug-disease prediction.
Target: **Baricitinib -> COVID-19** — expected to show multiple mechanistic pathways
through JAK1/JAK2 (cytokine storm) and AAK1 (viral endocytosis).

In [ ]:
program_explain = r'''
CREATE RULE ppi_reach AS
  MATCH (p:Protein)-[a:ASSOCIATED_WITH]->(dis:Disease)
  YIELD KEY p, KEY dis, a.gda_score AS strength

CREATE RULE ppi_reach AS
  MATCH (p:Protein)-[i:INTERACTS]->(p2:Protein)-[a:ASSOCIATED_WITH]->(dis:Disease)
  YIELD KEY p, KEY dis, i.string_score * a.gda_score AS strength

CREATE RULE ppi_reach AS
  MATCH (p:Protein)-[i1:INTERACTS]->(p2:Protein)-[i2:INTERACTS]->(p3:Protein)-[a:ASSOCIATED_WITH]->(dis:Disease)
  YIELD KEY p, KEY dis, i1.string_score * i2.string_score * a.gda_score AS strength

CREATE RULE drug_pathway AS
  MATCH (d:Drug)-[b:BINDS]->(p:Protein)
  WHERE p IS ppi_reach TO dis
  YIELD KEY d, KEY dis, b.confidence * strength AS pathway_score

CREATE RULE repurposing_signal AS
  MATCH (d:Drug)
  WHERE d IS drug_pathway TO dis
  FOLD evidence = MNOR(pathway_score)
  YIELD KEY d, KEY dis, evidence

EXPLAIN RULE repurposing_signal WHERE d.name = 'Baricitinib' AND dis.name = 'COVID-19'
'''

explain_out = session.locy_with(program_explain).with_config({'max_iterations': 200, 'timeout_secs': 60.0}).run()
explain_cmd = next(cmd for cmd in explain_out.command_results if cmd.command_type == 'explain')
tree = explain_cmd.tree
print('Derivation tree for Baricitinib -> COVID-19:')
_print_tree(tree, max_depth=4, max_children=5)


## 7) ASSUME — Counterfactual Binding Simulation

"What if Metformin also binds EGFR at high confidence?"

This temporarily adds a hypothetical BINDS edge and re-evaluates the repurposing signal.
After evaluation, the hypothetical edge is rolled back — no persistent graph mutation.

In [ ]:
assume_program = r'''
CREATE RULE ppi_reach AS
  MATCH (p:Protein)-[a:ASSOCIATED_WITH]->(dis:Disease)
  YIELD KEY p, KEY dis, a.gda_score AS strength

CREATE RULE ppi_reach AS
  MATCH (p:Protein)-[i:INTERACTS]->(p2:Protein)-[a:ASSOCIATED_WITH]->(dis:Disease)
  YIELD KEY p, KEY dis, i.string_score * a.gda_score AS strength

CREATE RULE ppi_reach AS
  MATCH (p:Protein)-[i1:INTERACTS]->(p2:Protein)-[i2:INTERACTS]->(p3:Protein)-[a:ASSOCIATED_WITH]->(dis:Disease)
  YIELD KEY p, KEY dis, i1.string_score * i2.string_score * a.gda_score AS strength

CREATE RULE drug_pathway AS
  MATCH (d:Drug)-[b:BINDS]->(p:Protein)
  WHERE p IS ppi_reach TO dis
  YIELD KEY d, KEY dis, b.confidence * strength AS pathway_score

CREATE RULE repurposing_signal AS
  MATCH (d:Drug)
  WHERE d IS drug_pathway TO dis
  FOLD evidence = MNOR(pathway_score)
  YIELD KEY d, KEY dis, evidence

ASSUME {
  MATCH (d:Drug {name: 'Metformin'}), (p:Protein {gene_symbol: 'EGFR'})
  CREATE (d)-[:BINDS {affinity_nm: 50.0, confidence: 0.85}]->(p)
} THEN {
  QUERY repurposing_signal WHERE d.name = 'Metformin' RETURN dis.name AS disease, evidence ORDER BY evidence DESC
}
'''

assume_out = session.locy_with(assume_program).with_config({'max_iterations': 200, 'timeout_secs': 60.0}).run()
assume_cmd = next(cmd for cmd in assume_out.command_results if cmd.command_type == 'assume')
assume_rows = assume_cmd.rows
print('Metformin repurposing signals with hypothetical EGFR binding:')
pprint(_norm_rows(assume_rows)[:10])

rollback_check = session.query("MATCH (:Drug {name: 'Metformin'})-[b:BINDS]->(:Protein {gene_symbol: 'EGFR'}) RETURN count(b) AS c")
print('Rollback check (should be 0):', rollback_check[0]['c'])


## 8) ABDUCE — Minimal Evidence Search

"What minimal binding evidence would make Metformin a candidate for Alzheimer disease?"

ABDUCE searches for the smallest set of graph modifications (new bindings, altered scores)
that would cause the target drug-disease pair to appear in the repurposing signal.

In [ ]:
program_abduce = r'''
CREATE RULE ppi_reach AS
  MATCH (p:Protein)-[a:ASSOCIATED_WITH]->(dis:Disease)
  YIELD KEY p, KEY dis, a.gda_score AS strength

CREATE RULE ppi_reach AS
  MATCH (p:Protein)-[i:INTERACTS]->(p2:Protein)-[a:ASSOCIATED_WITH]->(dis:Disease)
  YIELD KEY p, KEY dis, i.string_score * a.gda_score AS strength

CREATE RULE ppi_reach AS
  MATCH (p:Protein)-[i1:INTERACTS]->(p2:Protein)-[i2:INTERACTS]->(p3:Protein)-[a:ASSOCIATED_WITH]->(dis:Disease)
  YIELD KEY p, KEY dis, i1.string_score * i2.string_score * a.gda_score AS strength

CREATE RULE drug_pathway AS
  MATCH (d:Drug)-[b:BINDS]->(p:Protein)
  WHERE p IS ppi_reach TO dis
  YIELD KEY d, KEY dis, b.confidence * strength AS pathway_score

CREATE RULE repurposing_signal AS
  MATCH (d:Drug)
  WHERE d IS drug_pathway TO dis
  FOLD evidence = MNOR(pathway_score)
  YIELD KEY d, KEY dis, evidence

ABDUCE repurposing_signal WHERE d.name = 'Metformin' AND dis.name = 'Alzheimer disease'
'''

abduce_out = session.locy_with(program_abduce).with_config({'max_abduce_candidates': 120, 'max_abduce_results': 12, 'timeout_secs': 180.0}).run()
abduce_cmd = next(cmd for cmd in abduce_out.command_results if cmd.command_type == 'abduce')
mods = abduce_cmd.modifications
print('Abduced modifications for Metformin -> Alzheimer disease:')
for i, item in enumerate(mods[:8], start=1):
    print(f'\nCandidate #{i}')
    pprint(item)


## 9) What To Expect

- **Novel candidates**: Drug-Disease pairs with MNOR evidence scores in [0, 1], ordered by descending evidence. Known indications are excluded.
- **Baricitinib -> COVID-19**: The EXPLAIN RULE derivation tree should show multiple pathway derivations through JAK1/JAK2 (cytokine storm pathway) and AAK1 (viral endocytosis pathway).
- **ASSUME (Metformin + EGFR)**: Adds a hypothetical EGFR binding for Metformin and shows updated repurposing signal scores. The binding is rolled back after evaluation.
- **ABDUCE (Metformin -> Alzheimer disease)**: Finds minimal graph modifications (e.g., new protein bindings or PPI edges) that would make Metformin a repurposing candidate for Alzheimer disease.
- **Structural analogues**: Drug pairs with similar molecular fingerprints (via `similar_to`) that share target disease areas, filtered to analogy >= 0.5.

## 10) Build-Time Assertions

These checks keep notebook execution meaningful in CI/docs builds.

In [ ]:
assert novel_rows, 'Expected non-empty novel candidate rows'
assert analogue_rows, 'Expected non-empty structural analogue rows'
assert all(0.0 <= float(r['evidence']) <= 1.0 for r in novel_rows), 'MNOR scores must be in [0,1]'
assert tree, 'Expected EXPLAIN RULE to produce a derivation tree'
assert tree.get('children') or tree.get('rule'), 'Expected derivation tree to have structure'
assert assume_rows, 'Expected ASSUME to produce result rows'
assert rollback_check[0]['c'] == 0, 'Expected ASSUME rollback (no persistent EGFR binding)'
if not mods:
    print('Note: ABDUCE returned no modifications (may need higher timeout or different target)')
else:
    print(f'ABDUCE found {len(mods)} modifications')
print('All notebook assertions passed.')

## 11) Cleanup

Removes the temporary on-disk database created for this run.

In [ ]:
shutil.rmtree(DB_DIR, ignore_errors=True)
print('Cleaned up', DB_DIR)